In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from pathlib import Path
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
pbc = pd.read_csv('data/pbc2.csv')
ftpred = pd.read_csv('data/ft_db_cbcts_dummy_vit-t_ftdet.csv')

In [54]:
ftpred[(ftpred.subsplit == 'train')].time.max()

47.0

In [4]:
ftpred_train = ftpred[ftpred['subsplit'] == 'train']
ftpred_val = ftpred[ftpred['subsplit'] == 'val']
ftpred_test = ftpred[ftpred['subsplit'] == 'test']

In [37]:
class FTPredDataset(Dataset):
    def __init__(self,
                 df: pd.DataFrame,
                 crop_or_pad_to: int = 30,
                 transforms = None):
        self.df = df
        self.transform = transforms
        self.groups = [g for _, g in self.df.groupby("id")]
        self.crop_or_pad_to = crop_or_pad_to

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, idx):
        sample = self.groups[idx]

        if self.transform:
            sample = self.transform(sample)
        X = sample.drop(columns=["event", "event_at_time", "time", "start", "stop", "id", "split", "subsplit"]).values
        X = self.get_padded_features(X)
        Y = (sample['time'] - sample['stop']).values[-1] # get last
        D = sample['event_at_time'].values[-1] # get last
        return X, Y, D

    def get_padded_features(self, x):
        """Helper function to pad variable length RNN inputs with nans."""
        d = self.crop_or_pad_to
        difference = d - len(x)
        if difference < 0:
            return x[:d]
        pads = np.nan*np.ones((difference,) + x.shape[1:])
        return np.concatenate([x, pads])

class AugmentTimeVaryingSurvivalFeatures:
    def __init__(self,
                 only_censored_samples: bool = False,
                 probability: float = 0.5):
        self.rng = np.random.default_rng()
        self.only_censored_samples = only_censored_samples
        self.probability = probability

    def __call__(self, x: pd.DataFrame) -> pd.DataFrame:
        total_samples = len(x)
        if total_samples <= 1:
            return x
        if self.only_censored_samples and (1 in x['event_at_time'].unique()):
            return x
        # only sample with a certain probability
        if self.rng.random() > self.probability:
            return x
        new_length = self.rng.integers(1, total_samples)
        new_sample = x.iloc[:new_length].copy()
        return new_sample

In [46]:
train_data = FTPredDataset(ftpred_train) #, transforms=AugmentTimeVaryingSurvivalFeatures())
val_data = FTPredDataset(ftpred_val)
test_data = FTPredDataset(ftpred_test)

In [49]:
x, y, d = train_data[0]
print(x.shape, y, d)

(30, 192) 0.0 0


In [40]:
train_data.groups[0]

,id,start,stop,event,split,subsplit,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,feature_79,feature_80,feature_81,feature_82,feature_83,feature_84,feature_85,feature_86,feature_87,feature_88,feature_89,feature_90,feature_91,feature_92,feature_93,feature_94,feature_95,feature_96,feature_97,feature_98,feature_99,feature_100,feature_101,feature_102,feature_103,feature_104,feature_105,feature_106,feature_107,feature_108,feature_109,feature_110,feature_111,feature_112,feature_113,feature_114,feature_115,feature_116,feature_117,feature_118,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128,feature_129,feature_130,feature_131,feature_132,feature_133,feature_134,feature_135,feature_136,feature_137,feature_138,feature_139,feature_140,feature_141,feature_142,feature_143,feature_144,feature_145,feature_146,feature_147,feature_148,feature_149,feature_150,feature_151,feature_152,feature_153,feature_154,feature_155,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165,feature_166,feature_167,feature_168,feature_169,feature_170,feature_171,feature_172,feature_173,feature_174,feature_175,feature_176,feature_177,feature_178,feature_179,feature_180,feature_181,feature_182,feature_183,feature_184,feature_185,feature_186,feature_187,feature_188,feature_189,feature_190,feature_191,time,event_at_time
0,2247,0.0,1.0,0,train,train,-0.068892,0.434450,-0.475962,-0.606508,-1.107826,-0.516562,-1.426366,0.924110,0.315815,-0.290152,0.467812,-0.617799,0.011929,0.252888,-0.596915,-2.849330,-0.217827,-0.302398,-0.464257,-1.775085,-0.320602,0.488962,0.287560,0.725694,0.648309,0.507853,0.326476,0.390315,0.444491,-1.348959,-0.185466,-0.416936,-0.906970,2.383559,-0.072003,0.112769,1.149105,-0.427800,-0.142462,-0.311258,-0.000711,-0.463965,0.156178,-0.153701,0.546417,-0.743221,-0.392014,0.522309,0.502649,-0.439763,-0.502359,-0.240684,-0.155769,0.510055,-1.249786,-0.208171,-0.649825,-1.271306,0.293010,-0.122519,0.425848,-0.656036,-1.035086,0.024229,-0.415100,0.247042,0.736095,0.457277,1.118314,-0.138723,0.603962,-0.034964,1.124289,-0.190289,-1.545760,-0.392904,-0.180932,1.205222,1.606579,1.354515,0.790959,-0.042253,0.043080,-0.538366,-0.282173,-0.090835,0.142220,0.532247,0.345122,-0.325657,-0.896765,0.117699,0.725369,-0.416583,-0.971546,-0.040033,1.464501,0.603989,1.123547,-0.499343,-0.098019,-0.428140,0.989843,-0.371760,0.552297,0.497693,-1.010880,0.157130,-0.293085,0.245201,-0.332840,0.536084,-0.582569,-0.604443,0.712304,-0.183913,0.283048,0.711450,-0.356052,-0.125031,-0.060907,-0.543221,0.496483,-0.380315,-0.635010,-0.036659,0.059456,0.572905,0.005715,-0.704481,-0.274242,-1.866201,1.047939,0.329890,0.249523,0.297566,-0.600154,-0.381039,-1.452676,-1.021666,0.249725,-0.013913,0.199483,-0.499990,-0.072221,-0.370190,0.582528,-0.187887,0.454385,0.486713,-0.337155,-1.497586,-0.138716,-1.087387,-0.318607,-1.145454,-1.118870,0.556388,0.471356,-0.732965,0.355277,-0.643899,-0.403243,0.419948,-0.356891,0.862359,-0.191525,0.443402,0.466493,-0.576038,0.681238,0.377939,0.412992,0.011536,-0.091395,0.205965,9.901992,-0.355886,-0.866850,-0.393897,-

In [50]:
np.isnan(x[:, 0])

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False,  True,  True,  True,
        True,  True,  True])

In [55]:
for i in range(len(test_data)):
    x, y, d = test_data[i]
    print(f'{y=}, {d=}')

y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=0
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=0
y=0.0, d=1
y=0.0, d=0
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=0
y=0.0, d=1
y=0.0, d=0
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=0
y=0.0, d=1
y=0.0, d=0
y=0.0, d=1
y=0.0, d=0
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=0
y=0.0, d=1
y=0.0, d=0
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=0
y=0.0, d=1
y=0.0, d=1
y=0.0, d=0
y=0.0, d=1
y=0.0, d=0
y=0.0, d=0
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1
y=0.0, d=1


In [43]:
val_data.groups[0]

,id,start,stop,event,split,subsplit,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,feature_79,feature_80,feature_81,feature_82,feature_83,feature_84,feature_85,feature_86,feature_87,feature_88,feature_89,feature_90,feature_91,feature_92,feature_93,feature_94,feature_95,feature_96,feature_97,feature_98,feature_99,feature_100,feature_101,feature_102,feature_103,feature_104,feature_105,feature_106,feature_107,feature_108,feature_109,feature_110,feature_111,feature_112,feature_113,feature_114,feature_115,feature_116,feature_117,feature_118,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128,feature_129,feature_130,feature_131,feature_132,feature_133,feature_134,feature_135,feature_136,feature_137,feature_138,feature_139,feature_140,feature_141,feature_142,feature_143,feature_144,feature_145,feature_146,feature_147,feature_148,feature_149,feature_150,feature_151,feature_152,feature_153,feature_154,feature_155,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165,feature_166,feature_167,feature_168,feature_169,feature_170,feature_171,feature_172,feature_173,feature_174,feature_175,feature_176,feature_177,feature_178,feature_179,feature_180,feature_181,feature_182,feature_183,feature_184,feature_185,feature_186,feature_187,feature_188,feature_189,feature_190,feature_191,time,event_at_time
190,8627,0.0,1.0,0,train,val,-0.859018,-0.177306,-1.163738,0.320626,-1.174936,-1.169593,-0.346020,-0.339829,-0.044220,-0.059034,0.203406,-0.374147,0.173983,1.430195,-1.113985,-1.280219,0.464748,-2.684733,-0.374080,-1.316408,0.112944,-0.449070,0.294957,0.877408,0.481111,0.258377,0.896642,-1.287352,-0.251695,-0.126010,0.024280,0.727466,-0.334345,1.984853,-0.133523,0.746833,-0.147693,-0.830733,-0.069244,-0.372449,0.479776,-1.238468,0.758744,-0.115445,0.028310,-0.468833,0.013311,-1.140311,0.040105,-0.160757,0.687038,0.844174,-1.434333,-0.315567,0.536281,-0.288848,-0.384523,-1.484845,-0.104346,1.382517,0.070386,-0.878728,0.308806,-0.618762,0.464111,0.440040,0.428460,-0.614283,0.889544,0.413472,-0.228066,-0.055612,0.133410,-0.065442,-0.063663,0.604300,-1.424404,-0.452750,1.731072,0.726786,1.031908,0.907635,-0.108849,-0.174023,-0.533426,-1.253023,0.729307,1.665406,0.249445,-0.063060,-1.011266,-0.013548,0.621338,-0.470537,-0.822636,0.198597,1.989948,0.100286,0.895872,-1.143496,-0.799614,1.121794,0.487941,1.020969,0.374867,0.414829,-0.921939,0.024330,-1.016670,-0.297920,0.199856,0.012966,-0.743908,-0.125117,0.649327,0.892691,0.398268,0.696615,0.670353,0.326763,-0.547191,0.013241,0.369417,-0.169768,-0.450168,0.191805,-0.587214,1.605104,-0.452136,-0.178573,-0.605398,-2.045234,0.905912,0.067162,0.549107,0.047516,0.447382,-0.192406,-1.019897,1.642809,0.446881,0.686994,-0.461478,-0.998399,-0.037673,-0.381670,-0.144790,-0.628971,-0.572485,-0.187470,0.457140,-0.240670,0.373838,-1.670797,0.844585,-0.027137,-0.819803,0.176330,0.017611,-0.450589,-0.141568,0.502160,0.143707,0.426257,-0.716758,-0.034763,0.028764,0.246395,0.504120,-0.234584,-0.274723,-0.044447,0.195279,-0.669354,-0.046672,0.155547,9.264411,-1.955967,-0.845966,-0.354192,-0.1409

In [44]:
np.isnan(x[:, 0])

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True])

## Dev

In [61]:
def get_data_for_ddh(df):
    ids = df['id'].unique()
    X_list = []
    Y_list = []
    D_list = []
    for id in ids:
        df_id = df[df['id'] == id]
        X = df_id.drop(columns=["event", "event_at_time", "time", "start", "stop", "id", "split", "subsplit"]).values
        Y = (df_id['time'] - df_id['stop']).values
        # convert Y to int (number of days) to be compatible with the discretization step
        Y = Y.astype(int)
        D = df_id['event_at_time'].values
        X_list.append(X)
        Y_list.append(Y)
        D_list.append(D)
    return X_list, Y_list, D_list, ids

def discretize(t, split, split_time=None):
    """
        Discretize the survival horizon

        Args:
            t (List of Array): Time of events
            split (int): Number of bins
            split_time (List, optional): List of bins (must be same length than split). Defaults to None.

        Returns:
            List of Array: Disretized events time
    """
    if split_time is None:
        _, split_time = np.histogram(np.concatenate(t), split - 1)
    t_discretized = np.array([np.digitize(t_, split_time, right = True) - 1 for t_ in t], dtype = object)
    return t_discretized, split_time

def get_padded_features(x):
    """Helper function to pad variable length RNN inputs with nans."""
    d = max([len(x_) for x_ in x])
    padx = []
    for i in range(len(x)):
        pads = np.nan*np.ones((d - len(x[i]),) + x[i].shape[1:])
        padx.append(np.concatenate([x[i], pads]))
    return np.array(padx)

In [65]:
X_train_list, Y_train_list, D_train_list, ids_train = get_data_for_ddh(ftpred_train)
X_val_list, Y_val_list, D_val_list, ids_val = get_data_for_ddh(ftpred_val)
X_test_list, Y_test_list, D_test_list, ids_test = get_data_for_ddh(ftpred_test)

In [67]:
max([len(x) for x in X_train_list]), max([len(x) for x in X_val_list]), max([len(x) for x in X_test_list])

(33, 31, 33)

In [56]:
X_train_padded = torch.from_numpy(get_padded_features(X_train_list)).type(torch.float32).to(device)
last_Y_train = torch.tensor([_[-1] for _ in Y_train_list]).type(torch.float32).to(device)
D_train = torch.tensor([_[-1] for _ in D_train_list]).type(torch.float32).to(device)

In [59]:
len(last_Y_train), last_Y_train, ids_train[0]

(103,
 tensor([3., 1., 1., 1., 1., 1., 1., 1., 1., 2., 1., 1., 2., 4., 1., 5., 1., 1.,
         1., 1., 1., 4., 1., 1., 1., 1., 1., 1., 1., 3., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 2., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 3., 1., 1., 1., 2., 1., 1.,
         2., 2., 3., 1., 1., 1., 1., 1., 1., 2., 1., 1., 1., 2., 1., 1., 1., 1.,
         1., 1., 1., 1., 2., 1., 1., 3., 1., 1., 2., 1., 1.]),
 2247)

In [60]:
ftpred_train[ftpred_train['id'] == ids_train[0]]

,id,start,stop,event,split,subsplit,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47,feature_48,feature_49,feature_50,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60,feature_61,feature_62,feature_63,feature_64,feature_65,feature_66,feature_67,feature_68,feature_69,feature_70,feature_71,feature_72,feature_73,feature_74,feature_75,feature_76,feature_77,feature_78,feature_79,feature_80,feature_81,feature_82,feature_83,feature_84,feature_85,feature_86,feature_87,feature_88,feature_89,feature_90,feature_91,feature_92,feature_93,feature_94,feature_95,feature_96,feature_97,feature_98,feature_99,feature_100,feature_101,feature_102,feature_103,feature_104,feature_105,feature_106,feature_107,feature_108,feature_109,feature_110,feature_111,feature_112,feature_113,feature_114,feature_115,feature_116,feature_117,feature_118,feature_119,feature_120,feature_121,feature_122,feature_123,feature_124,feature_125,feature_126,feature_127,feature_128,feature_129,feature_130,feature_131,feature_132,feature_133,feature_134,feature_135,feature_136,feature_137,feature_138,feature_139,feature_140,feature_141,feature_142,feature_143,feature_144,feature_145,feature_146,feature_147,feature_148,feature_149,feature_150,feature_151,feature_152,feature_153,feature_154,feature_155,feature_156,feature_157,feature_158,feature_159,feature_160,feature_161,feature_162,feature_163,feature_164,feature_165,feature_166,feature_167,feature_168,feature_169,feature_170,feature_171,feature_172,feature_173,feature_174,feature_175,feature_176,feature_177,feature_178,feature_179,feature_180,feature_181,feature_182,feature_183,feature_184,feature_185,feature_186,feature_187,feature_188,feature_189,feature_190,feature_191,time,event_at_time
0,2247,0.0,1.0,0,train,train,-0.068892,0.434450,-0.475962,-0.606508,-1.107826,-0.516562,-1.426366,0.924110,0.315815,-0.290152,0.467812,-0.617799,0.011929,0.252888,-0.596915,-2.849330,-0.217827,-0.302398,-0.464257,-1.775085,-0.320602,0.488962,0.287560,0.725694,0.648309,0.507853,0.326476,0.390315,0.444491,-1.348959,-0.185466,-0.416936,-0.906970,2.383559,-0.072003,0.112769,1.149105,-0.427800,-0.142462,-0.311258,-0.000711,-0.463965,0.156178,-0.153701,0.546417,-0.743221,-0.392014,0.522309,0.502649,-0.439763,-0.502359,-0.240684,-0.155769,0.510055,-1.249786,-0.208171,-0.649825,-1.271306,0.293010,-0.122519,0.425848,-0.656036,-1.035086,0.024229,-0.415100,0.247042,0.736095,0.457277,1.118314,-0.138723,0.603962,-0.034964,1.124289,-0.190289,-1.545760,-0.392904,-0.180932,1.205222,1.606579,1.354515,0.790959,-0.042253,0.043080,-0.538366,-0.282173,-0.090835,0.142220,0.532247,0.345122,-0.325657,-0.896765,0.117699,0.725369,-0.416583,-0.971546,-0.040033,1.464501,0.603989,1.123547,-0.499343,-0.098019,-0.428140,0.989843,-0.371760,0.552297,0.497693,-1.010880,0.157130,-0.293085,0.245201,-0.332840,0.536084,-0.582569,-0.604443,0.712304,-0.183913,0.283048,0.711450,-0.356052,-0.125031,-0.060907,-0.543221,0.496483,-0.380315,-0.635010,-0.036659,0.059456,0.572905,0.005715,-0.704481,-0.274242,-1.866201,1.047939,0.329890,0.249523,0.297566,-0.600154,-0.381039,-1.452676,-1.021666,0.249725,-0.013913,0.199483,-0.499990,-0.072221,-0.370190,0.582528,-0.187887,0.454385,0.486713,-0.337155,-1.497586,-0.138716,-1.087387,-0.318607,-1.145454,-1.118870,0.556388,0.471356,-0.732965,0.355277,-0.643899,-0.403243,0.419948,-0.356891,0.862359,-0.191525,0.443402,0.466493,-0.576038,0.681238,0.377939,0.412992,0.011536,-0.091395,0.205965,9.901992,-0.355886,-0.866850,-0.393897,-

In [47]:
idx = 10
X_train_list[idx].shape, Y_train_list[idx], D_train_list[idx]

((14, 192),
 array([18, 17, 16, 15, 14, 11, 10,  9,  8,  7,  4,  3,  2,  1]),
 array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]))

In [35]:
# manually specify the duration grid
Y_train_stacked_np = np.concatenate(Y_train_list)
D_train_stacked_np = np.concatenate(D_train_list)
print(Y_train_stacked_np.shape, D_train_stacked_np.shape)
mask = (D_train_stacked_np >= 1)  # boolean mask specifying which training points were not censored
duration_grid_train_np = np.unique(Y_train_stacked_np[mask])

# the `discretize` function allows us to supply it with a pre-specified grid
Y_train_discrete_np, _ = \
    discretize(Y_train_list,
                len(duration_grid_train_np) - 1,
                duration_grid_train_np)
duration_grid_train_np

(1907,) (1907,)


array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
       14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26.,
       27., 28., 29., 30., 31., 32., 33., 35., 36.])

In [44]:
np.unique(np.concatenate(Y_test_list))

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
       14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26.,
       27., 28., 29., 30., 31., 32., 33., 34., 35., 36., 37., 38., 39.])

In [25]:
num_durations = 47
Y_train_discrete_np, duration_grid_train_np = discretize(Y_train_list, num_durations)
duration_grid_train_np

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
       14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26.,
       27., 28., 29., 30., 31., 32., 33., 34., 35., 36., 37., 38., 39.,
       40., 41., 42., 43., 44., 45., 46., 47.])

In [37]:
output_num_durations = len(duration_grid_train_np)
print(f'Number of discretized durations to be used with Dynamic-DeepHit: {output_num_durations}')
print('Duration grid:', duration_grid_train_np)

Number of discretized durations to be used with Dynamic-DeepHit: 35
Duration grid: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17. 18.
 19. 20. 21. 22. 23. 24. 25. 26. 27. 28. 29. 30. 31. 32. 33. 35. 36.]


In [40]:
Y_train_discrete_np.shape, len(Y_train_discrete_np[idx]), Y_train_discrete_np[idx]

((103,), 14, array([16, 15, 14, 13, 12,  9,  8,  7,  6,  5,  2,  1,  0, -1]))